# Allergen Label-Consistency Bench v1
Question: given only the finished label (text or photo), can a model tell whether the label itself is inconsistent with a required allergen declaration?
Three outcomes per label: `violation_supported`, `no_visible_defect`, `insufficient_evidence`. Two arms: transcribed text, and the image.

Data: FDA undeclared-allergen announcements (2019–2026, `fda_allergen_notices_2019_2026.json`), Open Food Facts for compliant negative controls.
All files live under `./local-data/allergen/`. Cells are self-contained and resumable. Run top to bottom; stop at the annotation gate (cell 3) and come back.

**Bring your own data.** No data ships with this notebook. Before cell 0, put these under `./local-data/allergen/`:
- `fda_allergen_notices_2019_2026.json`: the notices list. Cell 0a of `allergen-bench-v1.ipynb` builds it from fda.gov.
- `annotation_sheet.csv` (optional; cell 3 writes a blank one; supply it only if you skip cell 3, because cell 3
  overwrites this file): one row per label, as cell 3 writes it, plus the
  columns cell 5 reads if present: `recalled` (y/n), `defect_type`, `arms` (both, text_only or image_only), `defects`
  (`source:type` pairs, semicolon-separated), `item_flags` and `bucket`. Cell 5 accepts `outcome` values
  `violation_supported`, `no_visible_defect` and `insufficient_evidence`.

Cell 4 writes the Open Food Facts controls to `controls_off.json` only; nothing merges them into `annotation_sheet.csv`,
so add those rows to the sheet by hand.

In [ ]:
# Cell removed for the public copy: it copied the operator's own annotation sheet into ./local-data/allergen/. Build yours at the annotation gate (cell 3), or see the note at the top.

In [ ]:
# ── 0. setup ────────────────────────────────────────────────────────
import os, json, pathlib, subprocess, sys
from openai import OpenAI
VOL = pathlib.Path("./local-data"); A = VOL/"allergen"
for d in ["images","transcripts","runs","judged","results"]: (A/d).mkdir(parents=True, exist_ok=True)
oai = OpenAI(timeout=180, max_retries=0)
orx = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", timeout=180, max_retries=0)
for pkg in ["pillow","requests","pandas"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",pkg], check=False)
src = A/"fda_allergen_notices_2019_2026.json"
print("notices file:", src.stat().st_size if src.exists() else "MISSING — upload fda_allergen_notices_2019_2026.json to allergen/")
if src.exists():
    N = json.loads(src.read_text()); print(len(N), "notices;", sum(1 for r in N if r.get("n_img")), "with images")


In [ ]:
# ── 1. download FDA images at best resolution ───────────────────────
import json, re, io, time, requests, pathlib
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
A = pathlib.Path("./local-data/allergen"); IMG = A/"images"
N = json.loads((A/"fda_allergen_notices_2019_2026.json").read_text())
H = {"User-Agent":"Mozilla/5.0 Chrome/126.0"}
def slug(u): return re.sub(r"[^a-z0-9]+","-", u.split("/safety/recalls-market-withdrawals-safety-alerts/")[-1].lower())[:60]
def fetch(u):
    full = re.sub(r"/files/styles/[^/]+/public/", "/files/", u).split("?")[0]
    for cand in (full, u):
        try:
            b = requests.get(cand, headers=H, timeout=30)
            if b.status_code == 200 and len(b.content) > 2000 and b.content[:4] not in (b"<!DO", b"<htm"):
                return Image.open(io.BytesIO(b.content)).convert("RGB"), cand
        except Exception: pass
    return None, None
def work(r):
    out = []
    for i, u in enumerate(r.get("imgs", [])[:8]):
        p = IMG/f"{slug(r['url'])}__{i}.jpg"
        if p.exists():
            im = Image.open(p); out.append({"path": str(p), "w": im.size[0], "h": im.size[1], "src": "cached"}); continue
        im, src = fetch(u)
        if im is None: out.append({"url": u, "status": "fail"}); continue
        im.save(p, quality=92); out.append({"path": str(p), "w": im.size[0], "h": im.size[1], "src": src})
    r["images"] = out; return r
t0 = time.time()
with ThreadPoolExecutor(6) as ex: N = list(ex.map(work, [r for r in N if r.get("n_img")]))
(A/"notices_with_images.json").write_text(json.dumps(N, indent=1))
imgs = [i for r in N for i in r["images"] if i.get("path")]
big = sum(1 for i in imgs if max(i["w"], i["h"]) >= 1000)
print(f"{len(N)} notices, {len(imgs)} images saved, {big} at >=1000px, {time.time()-t0:.0f}s")


In [ ]:
# ── 2. transcribe every image with a vision model → text arm + readability ─
# Uses Gemini 3.8 Flash via OpenRouter (cheap, BYOK). Transcripts are reviewed by hand in cell 3 before use.
import os, json, base64, pathlib, time, random
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's listed price in
# USD per million tokens (input, output). Entries the repo does not state are left for you to fill.
from datetime import datetime, timezone
import math
# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v
RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)   # one cap per run; override with the RUN_BUDGET_USD environment variable
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# Confirmed listed prices, USD per million tokens (input, output), retrieved 2026-11-05. Each is the highest rate that
# could apply to a plain request. OpenRouter slugs: https://openrouter.ai/api/v1/models/<slug>/endpoints
# Direct OpenAI models (no slash): https://developers.openai.com/api/docs/pricing and /api/docs/models/<model>
PRICE = {
    "google/gemini-3.8-flash": (1.35, 6.75),
    "anthropic/claude-opus-5": (10, 50),
    "anthropic/claude-sonnet-5": (2.2, 11),
    "google/gemini-3.1-pro-preview": (7.2, 32.4),
    "moonshotai/kimi-k3": (4.5, 22.5),
    "x-ai/grok-4.6": (8, 24),
    "gpt-6-astra": (20, 75),
    "gpt-5.6-sol": (10, 45),
    "gpt-5.6-luna": (0.4, 1.8),
}
# Every model this notebook can call. A model with no confirmed price in PRICE stops the run before any paid call.
CALLABLE_MODELS = ["google/gemini-3.8-flash", "gpt-6-astra", "gpt-5.6-sol", "gpt-5.6-luna", "anthropic/claude-opus-5",
                   "anthropic/claude-sonnet-5", "google/gemini-3.1-pro-preview", "moonshotai/kimi-k3", "x-ai/grok-4.6"]
missing_prices = sorted(set(CALLABLE_MODELS) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"No confirmed listed price for {missing_prices}. Nothing was sent. Set an (input $/M, output $/M) "
                       "entry for each of these models in the PRICE table in this cell, from the provider's price page, then rerun.")
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
# One ledger root and one lock for every notebook: SPEND_LEDGER_DIR, default <repo root>/spend_ledger.
# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _spend_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent
SPEND_ROOT = _spend_root()
RUN_DIR = os.path.join(SPEND_ROOT, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(RUN_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. One exclusive lock file, SPEND_ROOT/runner.lock: a second runner
# refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died without
# exiting) is never cleared automatically: confirm no runner is alive, then delete SPEND_ROOT/runner.lock by hand.
import atexit
RUNNER_LOCK = os.path.join(SPEND_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be active, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


import json
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage, direct=False):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation. A direct call (direct=True) reports
    tokens and no cost: it settles at listed price times complete reported token counts, else at the worst case."""
    if direct and isinstance(usage, dict):
        i, o = usage.get("prompt_tokens"), usage.get("completion_tokens")
        if _is_count(i) and _is_count(o) and i > 0 and body["model"] in PRICE:
            pin, pout = PRICE[body["model"]]
            c = pin / 1e6 * i + pout / 1e6 * o
            if 0 < c < float("inf"):
                return c, "usage_tokens"
        return worst_case_cost(body), "worst_case"
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage, direct=False):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage, direct)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source != "worst_case" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def _sdk_text(kw):
    """The request text, as a bound on input size (every field except the model and the output cap)."""
    return json.dumps({k: v for k, v in kw.items() if k not in ("model", "modelId", "extra_body")}, default=str,
                      ensure_ascii=False)


def guarded_chat(client, purpose, **kw):
    """OpenRouter chat call through the SDK. provider.max_price and usage include go in extra_body; the attempt is
    reserved at its worst case first and settled to the cost OpenRouter reports."""
    body = openrouter_body({"model": kw["model"], "messages": kw["messages"], "max_tokens": kw.get("max_tokens")})
    eb = dict(kw.get("extra_body") or {})
    eb["provider"] = {**(eb.get("provider") or {}), "max_price": body["provider"]["max_price"]}
    eb["usage"] = {"include": True}
    kw = {**kw, "extra_body": eb}
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = client.chat.completions.create(**kw)
        try:
            usage = r.usage.model_dump() if r.usage is not None else None
        except Exception:
            usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


def guarded_responses(client, purpose, **kw):
    """Direct OpenAI Responses call (the user's own key). No OpenRouter fields apply. Reserved before sending at listed
    output price times max_output_tokens plus listed input price times a byte bound on the request; settled on the
    reported token counts when complete, else at the worst case. PRICE must list the model."""
    body = {"model": kw["model"], "messages": [{"role": "user", "content": _sdk_text(kw)}],
            "max_tokens": kw.get("max_output_tokens")}
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = client.responses.create(**kw)
        try:
            usage = {"prompt_tokens": r.usage.input_tokens, "completion_tokens": r.usage.output_tokens}
        except Exception:
            usage = None
        return r
    finally:
        ledger_settle(aid, body, usage, direct=True)


def guarded_converse(client, purpose, **kw):
    """Direct Bedrock Converse call (the user's own key). No OpenRouter fields apply. Reserved before sending like the
    Responses call; settled on complete reported token counts, else at the worst case. inferenceConfig.maxTokens
    bounds the output; PRICE must list the model id."""
    mt = (kw.get("inferenceConfig") or {}).get("maxTokens")
    body = {"model": kw["modelId"], "messages": [{"role": "user", "content": _sdk_text(kw)}], "max_tokens": mt}
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = client.converse(**kw)
        try:
            usage = {"prompt_tokens": r["usage"]["inputTokens"], "completion_tokens": r["usage"]["outputTokens"]}
        except Exception:
            usage = None
        return r
    finally:
        ledger_settle(aid, body, usage, direct=True)



orx = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", timeout=180, max_retries=0)
A = pathlib.Path("./local-data/allergen"); TR = A/"transcripts"
N = json.loads((A/"notices_with_images.json").read_text())
TRANSCRIBER = "google/gemini-3.8-flash"; PIN = {"provider": {"order": ["google-vertex/global"], "allow_fallbacks": False}}
PROMPT = """You are transcribing a food package photo. Return JSON only:
{"panel_visible": true|false, "readable": "full"|"partial"|"none",
 "ingredient_statement": "<verbatim text or empty>", "contains_statement": "<verbatim text or empty>",
 "may_contain_statement": "<verbatim text or empty>", "front_of_pack_text": "<product name and claims, brief>",
 "language": "<en|es|other>", "notes": "<anything cut off, blurry, or ambiguous>"}
Transcribe exactly what is printed. Do not infer allergens. Do not correct spelling. If no ingredient panel is visible, say so."""
def transcribe(path):
    out = TR/(pathlib.Path(path).stem + ".json")
    if out.exists(): return None
    b64 = base64.b64encode(pathlib.Path(path).read_bytes()).decode()
    for attempt in range(4):
        try:
            r = guarded_chat(orx, "call", model=TRANSCRIBER, temperature=0, max_tokens=2000, messages=[{"role":"user","content":[{"type":"text","text":PROMPT},
                              {"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{b64}"}}]}], extra_body={**PIN, "reasoning":{"effort":"low"}})
            raw = r.choices[0].message.content.strip().strip("`").removeprefix("json").strip()
            d = json.loads(raw); d["image"] = path; d["transcriber"] = r.model
            out.write_text(json.dumps(d, indent=1)); return d
        except Exception as e:
            err = str(e)[:200]; time.sleep(2**attempt + random.random())
    out.write_text(json.dumps({"image": path, "error": err})); return None
paths = [i["path"] for r in N for i in r["images"] if i.get("path")]
print(len(paths), "images;", sum(1 for p in paths if (TR/(pathlib.Path(p).stem+".json")).exists()), "already transcribed")
done = 0; t0 = time.time()
with ThreadPoolExecutor(6) as ex:
    for f in as_completed([ex.submit(transcribe, p) for p in paths]):
        done += 1
        if done % 50 == 0: print(f"  {done}/{len(paths)}  {time.time()-t0:.0f}s")
T = [json.loads(p.read_text()) for p in TR.glob("*.json")]
ok = [t for t in T if "error" not in t]
print(f"transcribed {len(ok)}; readable full={sum(t.get('readable')=='full' for t in ok)} partial={sum(t.get('readable')=='partial' for t in ok)} none={sum(t.get('readable')=='none' for t in ok)}")
print("images with an ingredient statement:", sum(bool(t.get("ingredient_statement")) for t in ok))


In [ ]:
# ── 3. build the annotation sheet (HUMAN GATE) ──────────────────────
# One row per notice with a readable panel. The annotator fills: outcome, allergens_visible, evidence_note, keep.
import json, pathlib, pandas as pd, re
A = pathlib.Path("./local-data/allergen"); TR = A/"transcripts"
N = json.loads((A/"notices_with_images.json").read_text())
T = {json.loads(p.read_text()).get("image"): json.loads(p.read_text()) for p in TR.glob("*.json")}
ALL = ["milk","egg","peanut","tree nut","almond","walnut","cashew","pecan","hazelnut","pistachio","macadamia","brazil nut","soy","wheat","fish","shellfish","shrimp","crab","sesame"]
rows = []
for r in N:
    ts = [T.get(i["path"]) for i in r["images"] if i.get("path") and T.get(i["path"])]
    best = max((t for t in ts if "error" not in t), key=lambda t: len(t.get("ingredient_statement","")), default=None)
    if not best or not best.get("ingredient_statement"): continue
    expected = [a for a in ALL if a in r["reason"].lower()]
    rows.append({"label_id": pathlib.Path(best["image"]).stem, "date": r["date"], "brand": r["brand"], "product": r["product"][:80],
                 "notice_url": r["url"], "image": best["image"], "readable": best.get("readable"), "language": best.get("language"),
                 "recall_reason": r["reason"], "expected_allergens": ";".join(expected),
                 "ingredient_statement": best.get("ingredient_statement",""), "contains_statement": best.get("contains_statement",""),
                 "may_contain": best.get("may_contain_statement",""), "transcriber_notes": best.get("notes",""),
                 # ---- fill by hand ----
                 "transcript_ok": "", "outcome": "", "allergens_visible": "", "evidence_note": "", "keep": ""})
df = pd.DataFrame(rows).sort_values("date", ascending=False)
df.to_csv(A/"annotation_sheet.csv", index=False)
print(len(df), "candidate labels written to allergen/annotation_sheet.csv")
print("""Fill per row:
  transcript_ok      y / n / fixed   (if 'fixed', correct ingredient_statement / contains_statement in place)
  outcome            violation_supported | no_visible_defect | insufficient_evidence   (judged from the label ONLY)
  allergens_visible  allergen(s) the label itself fails to declare, semicolon-separated; blank if none visible
  evidence_note      one line: the term that gives it away (e.g. 'whey, no milk in Contains'), or why it's insufficient
  keep               y / n
Aim for 60–100 kept rows. Then add compliant controls in cell 4.""")


In [ ]:
# ── 4. compliant negative controls from Open Food Facts ─────────────
# Pulls US products with an ingredient image and an explicit allergen tag list. These are NOT recalled; outcome = no_visible_defect unless annotation says otherwise.
import json, pathlib, requests, io, time
from PIL import Image
A = pathlib.Path("./local-data/allergen"); IMG = A/"images"; TR = A/"transcripts"
H = {"User-Agent": "labs-benchmark/1.0 (research; you@example.com)"}
N_CONTROLS = 40
url = "https://world.openfoodfacts.org/api/v2/search"
params = {"countries_tags_en": "united-states", "fields": "code,product_name,brands,ingredients_text_en,allergens_tags,traces_tags,image_ingredients_url,image_ingredients_small_url,lang",
          "page_size": 200, "sort_by": "unique_scans_n", "lc": "en"}
r = requests.get(url, params=params, headers=H, timeout=60).json()
cands = [p for p in r.get("products", []) if p.get("image_ingredients_url") and p.get("ingredients_text_en") and p.get("allergens_tags")]
print(len(cands), "OFF candidates with ingredient image + allergen tags")
ctrl = []
for p in cands:
    if len(ctrl) >= N_CONTROLS: break
    try:
        b = requests.get(p["image_ingredients_url"], headers=H, timeout=30).content
        im = Image.open(io.BytesIO(b)).convert("RGB")
        if max(im.size) < 600: continue
        path = IMG/f"off-{p['code']}__0.jpg"; im.save(path, quality=92)
        ctrl.append({"label_id": path.stem, "source": "openfoodfacts", "brand": p.get("brands",""), "product": p.get("product_name",""),
                     "off_code": p["code"], "image": str(path), "off_ingredients_text": p["ingredients_text_en"],
                     "off_allergens": ";".join(t.replace("en:","") for t in p["allergens_tags"]),
                     "off_traces": ";".join(t.replace("en:","") for t in p.get("traces_tags",[]))})
        time.sleep(0.3)
    except Exception as e: continue
(A/"controls_off.json").write_text(json.dumps(ctrl, indent=1))
print(len(ctrl), "controls saved to controls_off.json only; nothing merges them into annotation_sheet.csv, so add these rows to the sheet by hand and annotate them (outcome likely 'no_visible_defect')")


In [ ]:
# ── 5. freeze the item set from the annotated sheet ─────────────────
import json, pathlib, pandas as pd
A = pathlib.Path("./local-data/allergen")
df = pd.read_csv(A/"annotation_sheet.csv").fillna("")
for c in ["recalled","defect_type","arms","defects","item_flags","bucket"]:
    if c not in df: df[c] = ""
keep = df[df["keep"].astype(str).str.lower().str.startswith("y")]
assert len(keep) > 0, "no rows marked keep=y"
assert set(keep["outcome"]) <= {"violation_supported","no_visible_defect","insufficient_evidence"}, set(keep["outcome"])
items = []
for _, r in keep.iterrows():
    items.append({"label_id": r.label_id, "image": r.image, "language": r.language or "en",
                  "arms": r.arms or "both", "bucket": r.bucket or "core", "flags": r.item_flags,
                  "text": {"ingredient_statement": r.ingredient_statement, "contains_statement": r.contains_statement, "may_contain": r.may_contain},
                  "truth": {"outcome": r.outcome,
                            "allergens_visible": [a.strip() for a in str(r.allergens_visible).split(";") if a.strip()],
                            "defects": [{"source": d.split(":")[0], "type": d.split(":")[1]} for d in str(r.defects).split(";") if ":" in d],
                            "evidence_note": r.evidence_note,
                            "recalled": r.recalled or ("n" if str(r.recall_reason).startswith("CONTROL") else "y"),
                            "recall_reason": r.recall_reason, "expected_allergens": [a for a in str(r.expected_allergens).split(";") if a]}})
(A/"items.json").write_text(json.dumps({"created": pd.Timestamp.now().isoformat(), "n": len(items), "items": items}, indent=1))
print(len(items), "items frozen →", A/"items.json")
print(pd.crosstab([keep.bucket, keep.recalled], keep.outcome))
print("defect pairs:", keep[keep.outcome=="violation_supported"].defects.str.split(";").explode().str.split(":").str[1].value_counts().to_dict())

In [ ]:
# ── 6. grid runner: text arm + image arm, resumable ────────────────
import os, json, base64, pathlib, time, random, requests
from openai import OpenAI
oai = OpenAI(timeout=180, max_retries=0)
orx = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", timeout=180, max_retries=0)
A = pathlib.Path("./local-data/allergen"); RUNS = A/"runs"
ITEMS = json.loads((A/"items.json").read_text())["items"]
REPS = 2; ARMS = ["text", "image"]
FIELD = [  # (client, model, label, tags)  — vision-capable only
    ("oai", "gpt-6-astra",                 "astra",            None),
    ("oai", "gpt-5.6-sol",                 "sol",              None),
    ("oai", "gpt-5.6-luna",                "luna",             None),
    ("orx", "anthropic/claude-opus-5",     "opus-5",           ["amazon-bedrock"]),
    ("orx", "anthropic/claude-sonnet-5",   "sonnet-5",         ["amazon-bedrock/global"]),
    ("orx", "google/gemini-3.1-pro-preview","gemini-3.1-pro",  ["google-vertex/global"]),
    ("orx", "moonshotai/kimi-k3",          "kimi-k3",          ["digitalocean"]),
    ("orx", "x-ai/grok-4.6",               "grok-4.6",         ["amazon-bedrock/us-west-2"]),
]
EFFORT = "high"
def pin(tags): return {"provider": {"order": tags, "allow_fallbacks": False}} if tags else {}

SYSTEM = """You review food labels for allergen-declaration consistency under US FDA rules (FALCPA plus the FASTER Act: milk, egg, fish, crustacean shellfish, tree nuts, peanuts, wheat, soy, sesame). You see ONLY the label. You do not know the product's formulation, supplier, or recall history, and you cannot establish what is actually in the package.
Decide one outcome:
- violation_supported: the label itself supports a specific labeling defect. Types: undeclared (an allergen-derived ingredient such as whey, casein, albumin, tahini, or "flour" with no grain named, whose food source is never declared); contains_omission (a Contains statement exists and omits an allergen the ingredient list shows); specificity (the source is declared only by class, e.g. FISH or TREE NUTS, without the required species or nut type).
- no_visible_defect: enough relevant text is readable and no defect is supported. This is the answer for an internally consistent label even if the product name makes you suspicious; you cannot see the product.
- insufficient_evidence: missing, unreadable, cut-off, or ambiguous evidence prevents the assessment. An oil listed without "highly refined" is ambiguous, not a defect.
Never guess an allergen the label does not show. Return JSON only:
{"outcome": "violation_supported|no_visible_defect|insufficient_evidence", "defects": [{"source": "<milk|egg|fish|shellfish|tree nut|peanut|wheat|soy|sesame>", "type": "undeclared|contains_omission|specificity", "term": "<exact term on the label>"}], "readability": "full|partial|none"}
defects is an empty list unless outcome is violation_supported."""

def prompt_text(it):
    t = it["text"]
    return f"INGREDIENT STATEMENT:\n{t['ingredient_statement']}\n\nCONTAINS STATEMENT:\n{t['contains_statement'] or '(none printed)'}\n\nMAY CONTAIN / ADVISORY:\n{t['may_contain'] or '(none printed)'}\n\nLanguage: {it['language']}"

def run_id(label, arm, lid, rep): return f"{label}__{arm}__{lid}__r{rep}"

def call(client, model, label, tags, arm, it, rep):
    err = "unknown"
    for attempt in range(5):
        t0 = time.time()
        try:
            if arm == "text":
                content = prompt_text(it)
                if client == "oai":
                    r = guarded_responses(oai, "call", model=model, instructions=SYSTEM, input=content, reasoning={"effort":EFFORT}, max_output_tokens=1500, service_tier="default")
                    raw = r.output_text; mid = r.model; u = r.usage; usage = {"in": u.input_tokens, "out": u.output_tokens}
                else:
                    r = guarded_chat(orx, "call", model=model, messages=[{"role":"system","content":SYSTEM},{"role":"user","content":content}], max_tokens=1500, extra_body={"reasoning":{"effort":EFFORT}, **pin(tags)})
                    raw = r.choices[0].message.content or ""; mid = r.model; usage = {"in": r.usage.prompt_tokens, "out": r.usage.completion_tokens}
            else:
                b64 = base64.b64encode(pathlib.Path(it["image"]).read_bytes()).decode()
                if client == "oai":
                    r = guarded_responses(oai, "call", model=model, instructions=SYSTEM, reasoning={"effort":EFFORT}, max_output_tokens=1500, service_tier="default", input=[{"role":"user","content":[{"type":"input_text","text":"Review this label photo."},{"type":"input_image","image_url":f"data:image/jpeg;base64,{b64}"}]}])
                    raw = r.output_text; mid = r.model; u = r.usage; usage = {"in": u.input_tokens, "out": u.output_tokens}
                else:
                    r = guarded_chat(orx, "call", model=model, max_tokens=1500, extra_body={"reasoning":{"effort":EFFORT}, **pin(tags)}, messages=[{"role":"system","content":SYSTEM},{"role":"user","content":[{"type":"text","text":"Review this label photo."},{"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{b64}"}}]}])
                    raw = r.choices[0].message.content or ""; mid = r.model; usage = {"in": r.usage.prompt_tokens, "out": r.usage.completion_tokens}
            raw = raw.strip().strip("`").removeprefix("json").strip()
            try: parsed = json.loads(raw)
            except Exception: parsed = None
            return {"run_id": run_id(label,arm,it["label_id"],rep), "label": label, "model": mid, "arm": arm, "label_id": it["label_id"], "rep": rep,
                    "raw": raw, "parsed": parsed, "usage": usage, "wall_s": round(time.time()-t0,1), "retries": attempt}
        except Exception as e:
            err = str(e)[:300]
            if any(k in err for k in ["429","5","timeout","Timeout"]): time.sleep(min(60, 2**attempt + random.random()))
            else: break
    return {"run_id": run_id(label,arm,it["label_id"],rep), "label": label, "arm": arm, "label_id": it["label_id"], "rep": rep, "error": err}

done = {p.stem for p in RUNS.glob("*.json")}
for client, model, label, tags in FIELD:
    for arm in ARMS:
        # image_only items are skipped in the text arm; text_only would be skipped in the image arm
        todo = [(it, rep) for rep in range(1, REPS+1) for it in ITEMS
                if (it.get("arms","both") in ("both", f"{arm}_only")) and run_id(label,arm,it["label_id"],rep) not in done]
        if not todo: continue
        print(f"\n== {label} / {arm}: {len(todo)} calls"); errs = 0; t0 = time.time()
        for i, (it, rep) in enumerate(todo, 1):
            rec = call(client, model, label, tags, arm, it, rep)
            (RUNS/f"{rec['run_id']}.json").write_text(json.dumps(rec, indent=1))
            if "error" in rec:
                errs += 1; print("  ERROR", rec["run_id"], rec["error"][:90])
                if errs >= 5 and i <= 6: print("  aborting config"); break
            if i % 20 == 0 or i == len(todo): print(f"  {i}/{len(todo)}  {time.time()-t0:.0f}s")
print("\nruns on disk:", len(list(RUNS.glob("*.json"))))

In [ ]:
# ── 7. grade + rollup (deterministic; no judge needed) ──────────────
import json, pathlib, pandas as pd, numpy as np
A = pathlib.Path("./local-data/allergen")
ITEMS = {it["label_id"]: it for it in json.loads((A/"items.json").read_text())["items"]}

# allergen source synonyms → canonical big-9 name
SYN = {"tree nut":["tree nut","treenut","almond","walnut","cashew","pecan","hazelnut","filbert","pistachio","macadamia","brazil","pine nut"],
       "shellfish":["shellfish","crustacean","shrimp","crab","lobster"], "wheat":["wheat","gluten","flour"],
       "fish":["fish","salmon","cod","tuna","pollock","bonito","surimi","eel","anchov"], "milk":["milk","dairy"], "egg":["egg"], "soy":["soy"], "peanut":["peanut"], "sesame":["sesame","tahini"]}
def nsrc(x):
    x = str(x).lower().strip()
    for k, vs in SYN.items():
        if any(v in x for v in vs): return k
    return x
def pairs(ds): return {(nsrc(d.get("source","")), str(d.get("type","")).lower()) for d in (ds or []) if isinstance(d, dict)}

rows = []
for p in (A/"runs").glob("*.json"):
    r = json.loads(p.read_text()); it = ITEMS[r["label_id"]]; t = it["truth"]
    tp = pairs(t.get("defects")); tsrc = {s for s,_ in tp}
    base = {"label": r["label"], "arm": r["arm"], "label_id": r["label_id"], "rep": r["rep"], "truth": t["outcome"],
            "recalled": t["recalled"], "bucket": it.get("bucket","core"), "arms": it.get("arms","both"),
            "defect_types": ";".join(sorted({ty for _,ty in tp})) or "none", "flags": it.get("flags","")}
    pz = r.get("parsed") if "error" not in r else None
    if not pz:
        rows.append({**base, "pred": None, "parsed": False, "outcome_ok": False, "pair_ok": False, "source_ok": False, "false_alarm": False}); continue
    pp = pairs(pz.get("defects")); psrc = {s for s,_ in pp}
    rows.append({**base, "pred": pz.get("outcome"), "parsed": True,
                 "outcome_ok": pz.get("outcome") == t["outcome"],
                 "pair_ok": (pp == tp) if t["outcome"]=="violation_supported" else (not pp),      # source AND type must match
                 "source_ok": (psrc == tsrc) if t["outcome"]=="violation_supported" else (not psrc),  # partial-credit view
                 "false_alarm": bool(pp) and t["outcome"] != "violation_supported",
                 "in_tok": (r.get("usage") or {}).get("in"), "out_tok": (r.get("usage") or {}).get("out")})
df = pd.DataFrame(rows); df.to_csv(A/"judged"/"graded.csv", index=False)
core = df[df.bucket=="core"]   # headline numbers use the core set only

# headline table
g = core.groupby(["label","arm"]).agg(n=("outcome_ok","size"), outcome_acc=("outcome_ok","mean"), pair_acc=("pair_ok","mean"),
        source_only_acc=("source_ok","mean"), false_alarm=("false_alarm","mean"), parse_fail=("parsed", lambda s: 1-s.mean()), out_tok=("out_tok","mean")).round(3)
g["overall"] = ((g.outcome_acc + g.pair_acc)/2).round(3)
g = g.sort_values(["arm","overall"], ascending=[True,False]); g.to_csv(A/"results"/"summary.csv")
print("== CORE set, per model per arm (pair_acc = source AND defect type both right)\n", g.to_string())

# which defect types each model catches
v = core[core.truth=="violation_supported"]
print("\n== violations caught, by defect type (outcome_ok)")
print(v.pivot_table(index=["label","arm"], columns="defect_types", values="outcome_ok", aggfunc="mean").round(2).to_string())

# fabrication test: does a model invent a defect on a clean label, and does the product being recalled make it worse?
c = core[core.truth=="no_visible_defect"]
print("\n== false-alarm rate on no-visible-defect labels: recalled products vs controls")
print(c.pivot_table(index=["label","arm"], columns="recalled", values="false_alarm", aggfunc="mean").round(2).to_string())

# cost of vision: only items that ran in both arms
paired = core[core.arms=="both"]
w = paired.groupby(["label","arm"]).apply(lambda d: ((d.outcome_ok.mean()+d.pair_ok.mean())/2)).unstack("arm")
w["vision_gap"] = (w["text"] - w["image"]).round(3)
print("\n== paired items only (both arms), text minus image\n", w.round(3).to_string())

# unreadable-image probe, reported on its own line, never in the headline
probe = df[df.bucket=="unreadable_probe"]
if len(probe):
    print("\n== unreadable-image probe (reported separately): share answering insufficient_evidence")
    print(probe.groupby("label").apply(lambda d: (d.pred=="insufficient_evidence").mean()).round(2).to_string())

print("\nconfusion (core):\n", pd.crosstab(core[core.parsed].truth, core[core.parsed].pred))